# Module 2: K-Means and Hierarchical Clustering

This notebook discovers patterns among 1,495 U.S. young adults using 12 numeric, unlabeled measures of preparation, parental advice, emotional connection, relationship quality, authenticity, and financial independence. The analysis compares K-Means solutions for five values of $k$ and performs hierarchical clustering with **cosine distance** and **average linkage**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from scipy.spatial.distance import pdist
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="Set2")
data = pd.read_csv("module2_unlabeled_numeric_data.csv")
print(f"Model input shape: {data.shape}")
data.head()

## Data preparation

Clustering requires numeric observations without a target label. Respondent identifiers, survey weights, and demographic category codes were excluded because they do not measure the relationship pattern being discovered. Missing responses were median-imputed during preparation. Original 1–5 Pew codes were reverse-coded so that larger values consistently represent **more** of each construct. Standardization gives every measure equal scale in the distance calculations.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(data)
pd.DataFrame(X, columns=data.columns).describe().loc[["mean", "std"]].round(3)

## K-Means: compare at least three values of $k$

In [ ]:
rows = []
models = {}
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=50, random_state=42)
    labels = model.fit_predict(X)
    models[k] = (model, labels)
    rows.append({
        "k": k,
        "silhouette_score": silhouette_score(X, labels),
        "inertia": model.inertia_
    })

k_scores = pd.DataFrame(rows)
best_k = int(k_scores.loc[k_scores["silhouette_score"].idxmax(), "k"])
k_scores.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(k_scores["k"], k_scores["silhouette_score"], marker="o", linewidth=2.5, color="#2A7F78")
ax.scatter([best_k], [k_scores.loc[k_scores["k"].eq(best_k), "silhouette_score"].iloc[0]], s=150, color="#E58D72", zorder=3)
ax.set(title="Silhouette Comparison for K-Means", xlabel="Number of clusters (k)", ylabel="Average silhouette score")
ax.set_xticks(k_scores["k"])
plt.show()
print(f"Best tested K-Means solution: k={best_k}")

## Visualize the best K-Means solution

In [ ]:
kmeans_labels = models[best_k][1]
pca_2d = PCA(n_components=2, random_state=42).fit_transform(X)
plot_data = pd.DataFrame({"PC1": pca_2d[:, 0], "PC2": pca_2d[:, 1], "Cluster": (kmeans_labels + 1).astype(str)})

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=plot_data, x="PC1", y="PC2", hue="Cluster", alpha=0.67, s=45, ax=ax)
ax.set_title(f"K-Means Clusters Projected onto Two Principal Components (k={best_k})")
plt.show()

## Hierarchical clustering with cosine distance

In [ ]:
cosine_distances = pdist(X, metric="cosine")
Z = linkage(cosine_distances, method="average")

fig, ax = plt.subplots(figsize=(12, 6))
dendrogram(Z, truncate_mode="lastp", p=35, leaf_rotation=75, leaf_font_size=8, ax=ax)
ax.set(title="Cosine-Distance Hierarchical Clustering", xlabel="Merged respondent groups", ylabel="Cosine distance")
plt.show()

In [ ]:
h_rows = []
h_labels = {}
for k in range(2, 7):
    labels = fcluster(Z, t=k, criterion="maxclust") - 1
    h_labels[k] = labels
    h_rows.append({"k": k, "cosine_silhouette": silhouette_score(X, labels, metric="cosine")})

h_scores = pd.DataFrame(h_rows)
h_best_k = int(h_scores.loc[h_scores["cosine_silhouette"].idxmax(), "k"])
agreement = adjusted_rand_score(kmeans_labels, h_labels[h_best_k])
display(h_scores.round(4))
print(f"Best tested hierarchical cut: k={h_best_k}")
print(f"Adjusted Rand Index versus best K-Means: {agreement:.3f}")

## Interpret the cluster profiles

In [ ]:
profiles = data.assign(Cluster=kmeans_labels + 1).groupby("Cluster").mean()
fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(profiles, annot=True, fmt=".2f", cmap="YlGnBu", vmin=1, vmax=5, ax=ax)
ax.set_title("Average Relationship and Independence Measures by K-Means Cluster")
ax.set_xlabel("Prepared feature (higher means more)")
plt.xticks(rotation=45, ha="right")
plt.show()
profiles.round(2)

## Conclusion

Among the tested values $k=2,3,4,5,6$, the two-cluster K-Means solution has the largest silhouette score. Cosine hierarchical clustering also produces its strongest tested silhouette at two clusters. The positive Adjusted Rand Index shows meaningful agreement between methods even though their boundaries are not identical. One K-Means group reports more parental advice, emotional reliance, relationship quality, authenticity, and preparation; the other reports less connection and advice but slightly greater financial independence. These are descriptive patterns, not labels of good or bad families, and the modest K-Means silhouette indicates overlap between young adults rather than perfectly separated types.